In [6]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 52.7 MB/s eta 0:00:00


In [8]:
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans


#  DATASET: Today's News Headlines
headlines = [
    "Apple officially discontinues Mac Pro workstation",
    "Samsung launches Galaxy Book 6 AI PCs in India",
    "Apple iPhone 18 series expected to roll out in phases",
    "CEO replaced by AI at major multinational company",
    "PM Modi inaugurates new Noida International Airport",
    "IndiGo flight makes emergency landing at Delhi Airport",
    "Global oil prices surge after Strait of Hormuz blocked",
    "India turns to Russia for natural gas seeks US waiver",
    "Yemens Houthis launch ballistic missile attack",
    "Stock market experiences slight dip amid inflation fears"
]

In [9]:

#  PRE-PROCESSING & FEATURE ENGINEERING

sentences = [headline.lower().split() for headline in headlines]

# Training Word2Vec model
model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, epochs=2000)

def sent_vec(sentence):
    words = sentence.lower().split()
    vectors = [model.wv[word] for word in words if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(100)
    return np.mean(vectors, axis=0)

X = np.array([sent_vec(headline) for headline in headlines])

In [10]:

# COSINE SIMILARITY BETWEEN 3 SENTENCES

print("--- Cosine Similarity ---")

s1 = sent_vec(headlines[0])
s2 = sent_vec(headlines[1])
s3 = sent_vec(headlines[4])

sim_1_2 = cosine_similarity([s1], [s2])[0][0]
sim_1_3 = cosine_similarity([s1], [s3])[0][0]
sim_2_3 = cosine_similarity([s2], [s3])[0][0]

print(f"Sentence 1: '{headlines[0]}'")
print(f"Sentence 2: '{headlines[1]}'")
print(f"Sentence 3: '{headlines[4]}'\n")

print(f"Similarity (S1 & S2): {sim_1_2:.4f}")
print(f"Similarity (S1 & S3): {sim_1_3:.4f}")
print(f"Similarity (S2 & S3): {sim_2_3:.4f}\n")

--- Cosine Similarity ---
Sentence 1: 'Apple officially discontinues Mac Pro workstation'
Sentence 2: 'Samsung launches Galaxy Book 6 AI PCs in India'
Sentence 3: 'PM Modi inaugurates new Noida International Airport'

Similarity (S1 & S2): 0.3654
Similarity (S1 & S3): 0.4174
Similarity (S2 & S3): 0.2623



In [11]:

# TEXT CLASSIFICATION (SUPERVISED)

print("--- Text Classification (Logistic Regression) ---")
y = [0, 0, 0, 0, 1, 1, 1, 1, 1, 0]

classifier = LogisticRegression(random_state=42)
classifier.fit(X, y)

test_headline = "New smartphone with AI features launched"
test_vec = sent_vec(test_headline)
prediction = classifier.predict([test_vec])[0]

label_map = {0: "Technology/Business", 1: "Politics/World News"}
print(f"Test Headline: '{test_headline}'")
print(f"Predicted Category: {label_map[prediction]}\n")

--- Text Classification (Logistic Regression) ---
Test Headline: 'New smartphone with AI features launched'
Predicted Category: Technology/Business



In [12]:

#  TEXT CLUSTERING (UNSUPERVISED)

print("--- Text Clustering (K-Means) ---")
kmeans = KMeans(n_clusters=2, random_state=42, n_init='auto')
kmeans.fit(X)

cluster_labels = kmeans.labels_
cluster_groups = {0: [], 1: []}

for i, label in enumerate(cluster_labels):
    cluster_groups[label].append(headlines[i])

for cluster_id, items in cluster_groups.items():
    print(f"Cluster {cluster_id}:")
    for h in items:
        print(f" - {h}")
    print()

--- Text Clustering (K-Means) ---
Cluster 0:
 - Samsung launches Galaxy Book 6 AI PCs in India
 - CEO replaced by AI at major multinational company
 - PM Modi inaugurates new Noida International Airport
 - IndiGo flight makes emergency landing at Delhi Airport
 - Global oil prices surge after Strait of Hormuz blocked
 - Yemens Houthis launch ballistic missile attack
 - Stock market experiences slight dip amid inflation fears

Cluster 1:
 - Apple officially discontinues Mac Pro workstation
 - Apple iPhone 18 series expected to roll out in phases
 - India turns to Russia for natural gas seeks US waiver

